# Final experiments, part 3: significance tests, and a second dataset (rel-f1)

**1. Significance (rel-trial).** Paired bootstrap tests on the saved predictions, for every claim we make: label history for each model, text, tuning, RelGNN vs GraphSAGE, LightGBM vs the best GNN.

**2. Replication on rel-f1** (RelBench's Formula 1 database: drivers, races, constructors, results). Three tasks: `driver-dnf` (will the driver fail to finish a race next month?), `driver-top3` (qualify in the top 3?), `driver-position` (average finishing position over the next two months).

Why rel-f1 tests the mechanism: in rel-trial, how related past trials turned out sits many hops away from the trial being predicted. In rel-f1, a driver's past race results sit **one hop** away. If label history helps the GNN much less here, the finding generalizes as: *label history matters when it is out of the GNN's reach.*

Runs: LightGBM (table / aggregates / aggregates without history), GraphSAGE GNN with text ± history, RelGNN ± history; 3 seeds each. **About 2–3 hours** on an L4. Results go to `results_f1/` on Drive; rel-trial results stay in `results_final/`.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
TRIAL = f"{WORK}/results_final"
F1 = f"{WORK}/results_f1"
CACHE = "/content/cache"  # graph cache on the Colab disk, not Drive (it is large and rebuildable)
os.makedirs(F1, exist_ok=True)
F1_TASKS = "driver-dnf driver-top3 driver-position"

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (about 1–2 minutes). **Expect 26 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

## 1. Significance tests on rel-trial (about 10–15 minutes)

In [ ]:
!python scripts/run.py bootstrap --results {TRIAL}
from IPython.display import Markdown, display
display(Markdown(open(f"{TRIAL}/tables/bootstrap.md").read()))

## 2. rel-f1: explore the data (a few minutes, downloads the dataset)

In [ ]:
!python scripts/run.py explore --dataset rel-f1 --tasks {F1_TASKS} --results {F1}

## 3. rel-f1: LightGBM baselines (CPU, about 20 minutes)

In [ ]:
!python scripts/run.py lgbm --dataset rel-f1 --tasks {F1_TASKS} --features table agg agg_nohist --seeds 0 1 2 --results {F1}

## 4. rel-f1: GNN with text, ± label history (about 40 minutes)

In [ ]:
!python scripts/run.py rdl --dataset rel-f1 --tasks {F1_TASKS} --text glove --seeds 0 1 2 --cache_dir {CACHE} --results {F1}
!python scripts/run.py rdl --dataset rel-f1 --tasks {F1_TASKS} --text glove --history --seeds 0 1 2 --cache_dir {CACHE} --results {F1}

## 5. rel-f1: RelGNN (official per-task configs), ± label history (about 1 hour)

In [ ]:
!python scripts/run.py rdl --dataset rel-f1 --tasks {F1_TASKS} --arch relgnn --text glove --seeds 0 1 2 --cache_dir {CACHE} --results {F1}
!python scripts/run.py rdl --dataset rel-f1 --tasks {F1_TASKS} --arch relgnn --text glove --history --seeds 0 1 2 --cache_dir {CACHE} --results {F1}

## Fix (Nov 27): redo the rel-f1 runs hit by the pre-1970 date bug
Dates before 1970 broke the aggregate and label-history features (rel-f1 starts in 1950). This deletes only the four affected rel-f1 models; then run the **Resume** cell below. The table-only LightGBM and the GNNs without history are unaffected and are kept.

In [ ]:
BAD = ["lgbm_agg", "lgbm_agg_nohist", "rdl_sage_hist", "relgnn_hist"]
for task in F1_TASKS.split():
    for m in BAD:
        for kind in ("runs", "preds"):
            d = f"{F1}/{kind}/{task}/{m}"
            if os.path.exists(d):
                shutil.rmtree(d); print("removed", d)

## Resume: run only what is missing
If a step crashed or the session stopped, run the setup cells and then only this cell.

In [ ]:
STEPS = {
    "lgbm_table": "lgbm --features table", "lgbm_agg": "lgbm --features agg",
    "lgbm_agg_nohist": "lgbm --features agg_nohist",
    "rdl_sage": "rdl --text glove", "rdl_sage_hist": "rdl --text glove --history",
    "relgnn": "rdl --arch relgnn --text glove", "relgnn_hist": "rdl --arch relgnn --text glove --history",
}
for task in F1_TASKS.split():
    for name, cmd in STEPS.items():
        if os.path.exists(f"{F1}/runs/{task}/{name}/seed2.json"):
            print(f"done     {task:16s} {name}")
            continue
        print(f"running  {task:16s} {name}")
        !python scripts/run.py {cmd} --dataset rel-f1 --tasks {task} --seeds 0 1 2 --cache_dir {CACHE} --results {F1}

## 6. rel-f1 results and significance, then download everything

In [ ]:
!python scripts/run.py tables --results {F1}
!python scripts/run.py bootstrap --dataset rel-f1 --tasks {F1_TASKS} --results {F1}
display(Markdown(open(f"{F1}/tables/main.md").read()))
display(Markdown(open(f"{F1}/tables/bootstrap.md").read()))

In [ ]:
os.makedirs("/content/part3", exist_ok=True)
shutil.copytree(F1, "/content/part3/results_f1", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("embeddings"))
shutil.copytree(f"{TRIAL}/tables", "/content/part3/results_final_tables", dirs_exist_ok=True)
shutil.make_archive("/content/final_results_part3", "zip", "/content/part3", ".")
from google.colab import files
files.download("/content/final_results_part3.zip")